
# Факторный анализ расходов на резервы

Ноутбук работает **с уже подготовленным DataFrame `df_out`** и ничего не читает из Excel.

### Что ожидается в `df_out`

Для каждой отчетной даты должны быть уже заполнены:

- `задолженность_{дата}`
- `OD_{дата}`
- `НИ_{дата}`
- `ПФН_{дата}`
- `НВВ_{дата}`
- `рестра_{дата}`
- `обеспеченность_{дата}`
- `ГР_{дата}`
- `%рез_{дата}`

Также должен существовать постоянный столбец `Баланс/внебаланс`.

### Что рассчитывает ноутбук

Для каждой даты автоматически создаются:

- `ушла НИ_{дата}`
- `пришла НИ_{дата}`
- `ушел ПФН_{дата}`
- `пришел ПФН_{дата}`
- `ушло НВВ_{дата}`
- `пришло НВВ_{дата}`
- `изменение рестры_{дата}`
- `изменилась обеспеченность_{дата}`
- `ухудшение качества_{дата}`
- `проверка_{дата}`
- `переоценка_{дата}`
- `изменение портфеля_{дата}`

### База для факторного эффекта

Используется **`OD` предыдущего отчетного периода**:

\[
\text{изменение качества}
=
OD_{prev}
\times
\frac{\%рез_{prev} - \%рез_{current}}{100}
\]

Если значение нельзя рассчитать, числовой результат остается **0**, а причина записывается в `проверка_{дата}`.

> Формулы переоценки и изменения портфеля в переданном VBA-файле отсутствуют, поэтому соответствующие функции пока возвращают `0`. Когда формулы будут определены, менять нужно только две отдельные функции.


In [ ]:

import pandas as pd
import numpy as np
import re


# =============================================================================
# НАСТРОЙКИ
# =============================================================================

# Названия уже заполненных столбцов внутри каждого отчетного периода.
SOURCE_COLUMNS = {
    "debt": "задолженность",
    "od": "OD",
    "ni": "НИ",
    "pfn": "ПФН",
    "nvv": "НВВ",
    "restra": "рестра",
    "security": "обеспеченность",
    "group": "ГР",
    "rate": "%рез",
}

# Постоянный столбец с типом позиции.
BALANCE_COLUMN = "Баланс/внебаланс"

# Допустимая техническая погрешность при сравнении чисел.
EPSILON = 1e-7


# Факторные столбцы.
# Их сумма должна быть равна столбцу "ухудшение качества_{дата}".
FACTOR_COLUMNS = [
    "ушла НИ",
    "пришла НИ",
    "ушел ПФН",
    "пришел ПФН",
    "ушло НВВ",
    "пришло НВВ",
    "изменение рестры",
    "изменилась обеспеченность",
]


# Все расчетные столбцы, которые ноутбук создаст самостоятельно.
CALC_COLUMNS = FACTOR_COLUMNS + [
    "ухудшение качества",
    "проверка",
    "переоценка",
    "изменение портфеля",
]



## 1. Поиск отчетных дат и проверка структуры

Даты не нужно задавать руками. Они автоматически извлекаются из столбцов вида `задолженность_01.01.2026`.

Если какого-либо обязательного исходного столбца нет, выполнение останавливается сразу с понятным списком отсутствующих полей.


In [ ]:

def make_col(name, period):
    """Формирует имя столбца вида 'НИ_01.02.2026'."""
    return f"{name}_{period}"


def find_periods(df):
    """Находит все отчетные даты по названиям столбцов задолженности."""

    pattern = re.compile(
        rf"^{re.escape(SOURCE_COLUMNS['debt'])}_(\d{{2}}\.\d{{2}}\.\d{{4}})$"
    )

    periods = []

    for column in df.columns:
        match = pattern.match(str(column))

        if match:
            periods.append(match.group(1))

    # Убираем дубли и сортируем как даты, а не как строки.
    periods = sorted(
        set(periods),
        key=lambda x: pd.to_datetime(x, format="%d.%m.%Y"),
    )

    return periods


# -----------------------------------------------------------------------------
# df_out должен быть создан ДО запуска этой ячейки.
# -----------------------------------------------------------------------------

if "df_out" not in globals():
    raise NameError(
        "DataFrame df_out не найден. "
        "Сначала сформируй df_out, затем запускай этот ноутбук."
    )


PERIODS = find_periods(df_out)

if len(PERIODS) == 0:
    raise ValueError(
        "Не найдены столбцы вида 'задолженность_01.01.2026'."
    )


# Проверяем обязательные исходные столбцы.
missing_columns = []

for period in PERIODS:
    for source_column in SOURCE_COLUMNS.values():
        column_name = make_col(source_column, period)

        if column_name not in df_out.columns:
            missing_columns.append(column_name)


if BALANCE_COLUMN not in df_out.columns:
    missing_columns.append(BALANCE_COLUMN)


if missing_columns:
    raise KeyError(
        "В df_out отсутствуют обязательные столбцы:\n"
        + "\n".join(missing_columns)
    )


# Создаем недостающие расчетные столбцы.
# В отличие от предыдущих версий, никаких NaN:
# числовые расчетные показатели сразу инициализируются нулем.
for period in PERIODS:
    for column in CALC_COLUMNS:

        full_column_name = make_col(column, period)

        if full_column_name not in df_out.columns:

            # "проверка" может содержать либо 0, либо текст замечания.
            if column == "проверка":
                df_out[full_column_name] = 0
            else:
                df_out[full_column_name] = 0.0


print("Найдены отчетные периоды:")
print(PERIODS)
print()
print(f"Количество строк df_out: {len(df_out):,}")



## 2. Технические функции

Эти функции приводят значения к единому виду, распознают признаки НИ/ПФН/НВВ/рестры и безопасно сравнивают группы риска.


In [ ]:

def to_float(value):
    """
    Безопасно преобразует значение в float.

    Поддерживает:
    5
    "5"
    "5,0"
    " 5 "

    Если преобразование невозможно — возвращает np.nan.
    np.nan здесь используется только как технический внутренний маркер;
    в итоговые расчетные столбцы он не записывается.
    """

    if pd.isna(value):
        return np.nan

    try:
        value = (
            str(value)
            .strip()
            .replace("\xa0", "")
            .replace(" ", "")
            .replace(",", ".")
        )

        return float(value)

    except (ValueError, TypeError):
        return np.nan


def normalize_scalar(value):
    """Нормализует текст для устойчивого сравнения."""

    if pd.isna(value):
        return None

    return (
        str(value)
        .strip()
        .lower()
        .replace("ё", "е")
    )


def is_active(value):
    """
    Переводит НИ / ПФН / НВВ в True / False.

    Основной ожидаемый формат:
    0 -> признака нет
    1 -> признак есть

    Дополнительно понимает 'да', '+', True, 'есть'.
    """

    if pd.isna(value):
        return False

    numeric = to_float(value)

    if not pd.isna(numeric):
        return numeric == 1

    text = normalize_scalar(value)

    return text in {
        "да",
        "true",
        "есть",
        "yes",
        "+",
    }


def contains_restra(value):
    """Определяет наличие реструктуризации / рестры."""

    if pd.isna(value):
        return False

    if is_active(value):
        return True

    text = normalize_scalar(value)

    if text is None:
        return False

    return (
        "рестр" in text
        or
        "реестр" in text
    )


def same_group(first_group, second_group):
    """Сравнивает группы риска: 2, 2.0 и '2' считаются одной ГР."""

    first_numeric = to_float(first_group)
    second_numeric = to_float(second_group)

    if (
        not pd.isna(first_numeric)
        and
        not pd.isna(second_numeric)
    ):
        return first_numeric == second_numeric

    return (
        normalize_scalar(first_group)
        ==
        normalize_scalar(second_group)
    )



## 3. Ставки промежуточных ГР — только из `df_out`

В коде **нет словаря ставок резервирования**.

Для сложных случаев, когда одновременно меняются несколько факторов, иногда нужна ставка промежуточной ГР. Она определяется по фактически встречающимся в `df_out` сочетаниям:

`Баланс/внебаланс + ГР -> %рез`

Если в `df_out` для одного и того же сочетания обнаружатся разные ставки, выполнение остановится — это защищает расчет от незаметного использования неоднозначного процента.


In [ ]:

# Справочник строится автоматически ИЗ САМОГО df_out.
rate_lookup = {}

# В этот список попадут противоречия, если они неожиданно встретятся.
rate_lookup_conflicts = []


for period in PERIODS:

    group_column = make_col(
        SOURCE_COLUMNS["group"],
        period,
    )

    rate_column = make_col(
        SOURCE_COLUMNS["rate"],
        period,
    )

    for index in df_out.index:

        balance_type = normalize_scalar(
            df_out.at[index, BALANCE_COLUMN]
        )

        risk_group = to_float(
            df_out.at[index, group_column]
        )

        reserve_rate = to_float(
            df_out.at[index, rate_column]
        )


        # По условию ГР и %рез заполнены для каждой строки.
        # Проверка ниже оставлена как техническая страховка.
        if (
            balance_type is None
            or pd.isna(risk_group)
            or pd.isna(reserve_rate)
        ):
            continue


        key = (
            balance_type,
            risk_group,
        )


        if key not in rate_lookup:

            rate_lookup[key] = reserve_rate

        else:

            existing_rate = rate_lookup[key]

            if abs(existing_rate - reserve_rate) > EPSILON:

                rate_lookup_conflicts.append(
                    {
                        "index": index,
                        "period": period,
                        "Баланс/внебаланс": balance_type,
                        "ГР": risk_group,
                        "ставка_1": existing_rate,
                        "ставка_2": reserve_rate,
                    }
                )


# Если одна ГР для одного типа позиции имеет разные ставки,
# нельзя однозначно рассчитать промежуточный эффект.
if rate_lookup_conflicts:

    rate_lookup_conflicts = pd.DataFrame(
        rate_lookup_conflicts
    )

    raise ValueError(
        "В df_out найдены разные %рез для одинаковых "
        "сочетаний Баланс/внебаланс + ГР:\n\n"
        f"{rate_lookup_conflicts}"
    )


def get_intermediate_rate(
    risk_group,
    balance_type,
    prev_group,
    prev_rate,
    current_group,
    current_rate,
):
    """
    Возвращает %рез для промежуточной ГР.

    1. Если ГР совпала со старой фактической — берем старый фактический %рез.
    2. Если ГР совпала с новой фактической — берем новый фактический %рез.
    3. Иначе берем ставку из rate_lookup, построенного из df_out.
    """

    if same_group(risk_group, prev_group):
        return prev_rate

    if same_group(risk_group, current_group):
        return current_rate


    balance_type = normalize_scalar(balance_type)
    risk_group = to_float(risk_group)


    if (
        balance_type is None
        or pd.isna(risk_group)
    ):
        return np.nan


    return rate_lookup.get(
        (balance_type, risk_group),
        np.nan,
    )


print(f"Найдено сочетаний Баланс/внебаланс + ГР: {len(rate_lookup)}")



## 4. Логика промежуточной группы риска

Этот блок нужен только тогда, когда между двумя датами изменилось **два или более факторов**.

Последовательность основана на логике исходного VBA:

- выход из рестры обрабатывается первым;
- вход в рестру — последним;
- среди НИ, ПФН, НВВ и обеспеченности сначала ищется фактор, который переводит договор ровно на следующую ГР в направлении фактической;
- если такого фактора нет, выбирается фактор, который сильнее приближает промежуточную ГР к итоговой;
- НВВ в исходной логике временно трактуется аналогично НИ.


In [ ]:

def calculate_ordinary_risk_group(
    ni,
    pfn,
    nvv,
    security,
):
    """
    Определяет ПРОМЕЖУТОЧНУЮ ГР для факторного анализа.

    Это не заменяет фактическую ГР из df_out.
    Функция нужна только для моделирования последовательных переходов,
    когда за один период изменилось несколько признаков.
    """

    security = normalize_scalar(security)

    if security is None:
        return np.nan


    # Высококачественное обеспечение -> 1 ГР.
    if "высококачествен" in security:
        return 1


    # Обеспеченный:
    # ПФН -> 3 ГР
    # НИ / НВВ -> 2 ГР
    # иначе -> 1 ГР
    if security == "обеспеченный":

        if pfn:
            return 3

        if ni or nvv:
            return 2

        return 1


    # Недостаточно обеспеченный:
    # ПФН -> 3 ГР
    # иначе -> 2 ГР
    if (
        "недостаточно" in security
        and
        "обеспеч" in security
    ):

        if pfn:
            return 3

        return 2


    # Не обеспеченный:
    # ПФН -> 4 ГР
    # НИ / НВВ -> 3 ГР
    # иначе -> 2 ГР
    if security in {
        "не обеспеченный",
        "необеспеченный",
    }:

        if pfn:
            return 4

        if ni or nvv:
            return 3

        return 2


    return np.nan


def apply_factor_to_state(
    factor,
    state,
    current_state,
):
    """Применяет к промежуточному состоянию только один изменившийся фактор."""

    new_state = state.copy()

    if factor == "NI":
        new_state["NI"] = current_state["NI"]

    elif factor == "PFN":
        new_state["PFN"] = current_state["PFN"]

    elif factor == "NVV":
        new_state["NVV"] = current_state["NVV"]

    elif factor == "REESTR":
        new_state["REESTR"] = current_state["REESTR"]

    elif factor == "SECURITY":
        new_state["SECURITY"] = current_state["SECURITY"]

    return new_state


def group_after_applying_factor(
    factor,
    state,
    current_state,
):
    """Определяет промежуточную ГР после применения одного фактора."""

    temp_state = apply_factor_to_state(
        factor=factor,
        state=state,
        current_state=current_state,
    )


    # Для состояния в рестре обычная матрица ГР не применяется.
    if temp_state["REESTR"]:
        return np.nan


    return calculate_ordinary_risk_group(
        ni=temp_state["NI"],
        pfn=temp_state["PFN"],
        nvv=temp_state["NVV"],
        security=temp_state["SECURITY"],
    )


def select_next_factor_index(
    pending_factors,
    state_group,
    final_group,
    state,
    current_state,
):
    """
    Выбирает следующий фактор при одновременном изменении нескольких признаков.

    Приоритет:
    1. Фактор, дающий переход ровно на следующую ГР к итоговой.
    2. Фактор, максимально приближающий ГР к итоговой.
    3. Если определить невозможно — исходный порядок факторов.
    """

    state_group = to_float(state_group)
    final_group = to_float(final_group)


    # Если ГР нечисловая, берем первый обычный фактор.
    if (
        pd.isna(state_group)
        or pd.isna(final_group)
    ):

        for i, factor in enumerate(pending_factors):

            if factor != "REESTR":
                return i

        return 0


    # Направление изменения ГР.
    if final_group > state_group:
        direction = 1

    elif final_group < state_group:
        direction = -1

    else:
        return 0


    desired_group = (
        state_group
        +
        direction
    )


    # -------------------------------------------------------------------------
    # 1. Ищем фактор, который дает ровно следующую ГР.
    # -------------------------------------------------------------------------

    for i, factor in enumerate(pending_factors):

        if factor == "REESTR":
            continue


        candidate_group = group_after_applying_factor(
            factor=factor,
            state=state,
            current_state=current_state,
        )

        candidate_group = to_float(candidate_group)


        if pd.isna(candidate_group):
            continue


        if candidate_group == desired_group:
            return i


    # -------------------------------------------------------------------------
    # 2. Ищем фактор, который максимально приближает ГР к фактической.
    # -------------------------------------------------------------------------

    current_distance = abs(
        final_group
        -
        state_group
    )

    best_index = None
    best_distance = np.inf


    for i, factor in enumerate(pending_factors):

        if factor == "REESTR":
            continue


        candidate_group = group_after_applying_factor(
            factor=factor,
            state=state,
            current_state=current_state,
        )

        candidate_group = to_float(candidate_group)


        if pd.isna(candidate_group):
            continue


        distance = abs(
            final_group
            -
            candidate_group
        )


        if (
            distance < current_distance
            and distance < best_distance
        ):

            best_distance = distance
            best_index = i


    if best_index is not None:
        return best_index


    # -------------------------------------------------------------------------
    # 3. Если лучший переход определить нельзя — сохраняем порядок.
    # -------------------------------------------------------------------------

    for i, factor in enumerate(pending_factors):

        if factor != "REESTR":
            return i


    return 0



## 5. Запись факторных эффектов и контроль знаков


In [ ]:

def add_contribution(
    output,
    factor,
    amount,
    prev_state,
    current_state,
):
    """Записывает рассчитанный эффект в нужный факторный столбец."""

    if factor == "NI":

        if (
            prev_state["NI"]
            and
            not current_state["NI"]
        ):
            output["ушла НИ"] += amount

        else:
            output["пришла НИ"] += amount


    elif factor == "PFN":

        if (
            prev_state["PFN"]
            and
            not current_state["PFN"]
        ):
            output["ушел ПФН"] += amount

        else:
            output["пришел ПФН"] += amount


    elif factor == "NVV":

        if (
            prev_state["NVV"]
            and
            not current_state["NVV"]
        ):
            output["ушло НВВ"] += amount

        else:
            output["пришло НВВ"] += amount


    elif factor == "REESTR":

        output["изменение рестры"] += amount


    elif factor == "SECURITY":

        output["изменилась обеспеченность"] += amount


def add_status(messages, message):
    """Добавляет уникальное сообщение в список замечаний."""

    if message not in messages:
        messages.append(message)


def validate_factor_signs(
    output,
    messages,
):
    """Проверяет ожидаемый знак прихода / ухода негативных факторов."""

    if output["ушла НИ"] < -EPSILON:
        add_status(
            messages,
            "Ушла НИ, но эффект отрицательный",
        )

    if output["пришла НИ"] > EPSILON:
        add_status(
            messages,
            "Пришла НИ, но эффект положительный",
        )

    if output["ушел ПФН"] < -EPSILON:
        add_status(
            messages,
            "Ушел ПФН, но эффект отрицательный",
        )

    if output["пришел ПФН"] > EPSILON:
        add_status(
            messages,
            "Пришел ПФН, но эффект положительный",
        )

    if output["ушло НВВ"] < -EPSILON:
        add_status(
            messages,
            "Ушло НВВ, но эффект отрицательный",
        )

    if output["пришло НВВ"] > EPSILON:
        add_status(
            messages,
            "Пришло НВВ, но эффект положительный",
        )



## 6. Расчет изменения качества для одной строки

Главное уточнение этой версии:

\[
\text{факторный эффект}
=
OD_{prev}
\times
\frac{\text{ставка до фактора} - \text{ставка после фактора}}{100}
\]

То есть **`задолженность` в факторных расчетах не используется**.

Если меняется один фактор — весь эффект относится на него.  
Если меняется два или более — строится последовательность промежуточных состояний.  
Последний фактор всегда доводит расчет до фактического `%рез` текущего периода.


In [ ]:

def calculate_quality_factors_for_row(
    row,
    old_period,
    new_period,
):
    """Рассчитывает факторный анализ между двумя соседними отчетными датами."""

    # -------------------------------------------------------------------------
    # Все числовые результаты по умолчанию равны 0.
    # Поэтому даже при невозможности расчета NaN в расчетные поля не попадет.
    # -------------------------------------------------------------------------

    output = {
        "ушла НИ": 0.0,
        "пришла НИ": 0.0,
        "ушел ПФН": 0.0,
        "пришел ПФН": 0.0,
        "ушло НВВ": 0.0,
        "пришло НВВ": 0.0,
        "изменение рестры": 0.0,
        "изменилась обеспеченность": 0.0,
        "ухудшение качества": 0.0,
        "проверка": 0,
    }

    messages = []


    # =========================================================================
    # OD ПРЕДЫДУЩЕГО ПЕРИОДА — БАЗА ДЛЯ ВСЕХ ФАКТОРНЫХ ЭФФЕКТОВ
    # =========================================================================

    od = to_float(
        row[
            make_col(
                SOURCE_COLUMNS["od"],
                old_period,
            )
        ]
    )


    # Фактические ГР.
    prev_group = row[
        make_col(
            SOURCE_COLUMNS["group"],
            old_period,
        )
    ]

    current_group = row[
        make_col(
            SOURCE_COLUMNS["group"],
            new_period,
        )
    ]


    # Фактические проценты резервирования напрямую из df_out.
    prev_rate = to_float(
        row[
            make_col(
                SOURCE_COLUMNS["rate"],
                old_period,
            )
        ]
    )

    current_rate = to_float(
        row[
            make_col(
                SOURCE_COLUMNS["rate"],
                new_period,
            )
        ]
    )


    # =========================================================================
    # ТЕХНИЧЕСКИЕ ПРОВЕРКИ
    # =========================================================================
    #
    # Если что-то неожиданно отсутствует, расчетные значения остаются 0,
    # а причина записывается в "проверка".
    # =========================================================================

    if pd.isna(od):

        output["проверка"] = (
            "Не заполнен OD предыдущего периода"
        )

        return pd.Series(output)


    if pd.isna(prev_rate):

        output["проверка"] = (
            "Не заполнен %рез предыдущего периода"
        )

        return pd.Series(output)


    if pd.isna(current_rate):

        output["проверка"] = (
            "Не заполнен %рез текущего периода"
        )

        return pd.Series(output)


    # =========================================================================
    # ОБЩЕЕ ИЗМЕНЕНИЕ КАЧЕСТВА
    # =========================================================================

    total_effect = (
        od
        *
        (
            prev_rate
            -
            current_rate
        )
        /
        100
    )

    output[
        "ухудшение качества"
    ] = total_effect


    # =========================================================================
    # СОСТОЯНИЕ ФАКТОРОВ В ПРЕДЫДУЩЕМ ПЕРИОДЕ
    # =========================================================================

    prev_state = {

        "NI": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["ni"],
                    old_period,
                )
            ]
        ),

        "PFN": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["pfn"],
                    old_period,
                )
            ]
        ),

        "NVV": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["nvv"],
                    old_period,
                )
            ]
        ),

        "REESTR": contains_restra(
            row[
                make_col(
                    SOURCE_COLUMNS["restra"],
                    old_period,
                )
            ]
        ),

        "SECURITY": row[
            make_col(
                SOURCE_COLUMNS["security"],
                old_period,
            )
        ],
    }


    # =========================================================================
    # СОСТОЯНИЕ ФАКТОРОВ В ТЕКУЩЕМ ПЕРИОДЕ
    # =========================================================================

    current_state = {

        "NI": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["ni"],
                    new_period,
                )
            ]
        ),

        "PFN": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["pfn"],
                    new_period,
                )
            ]
        ),

        "NVV": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["nvv"],
                    new_period,
                )
            ]
        ),

        "REESTR": contains_restra(
            row[
                make_col(
                    SOURCE_COLUMNS["restra"],
                    new_period,
                )
            ]
        ),

        "SECURITY": row[
            make_col(
                SOURCE_COLUMNS["security"],
                new_period,
            )
        ],
    }


    # =========================================================================
    # ИЩЕМ ИЗМЕНИВШИЕСЯ ФАКТОРЫ
    # =========================================================================

    changed_factors = []


    if prev_state["NI"] != current_state["NI"]:
        changed_factors.append("NI")

    if prev_state["PFN"] != current_state["PFN"]:
        changed_factors.append("PFN")

    if prev_state["NVV"] != current_state["NVV"]:
        changed_factors.append("NVV")

    if prev_state["REESTR"] != current_state["REESTR"]:
        changed_factors.append("REESTR")

    if (
        normalize_scalar(prev_state["SECURITY"])
        !=
        normalize_scalar(current_state["SECURITY"])
    ):
        changed_factors.append("SECURITY")


    # =========================================================================
    # ЕСЛИ ОБЩИЙ ЭФФЕКТ РАВЕН 0
    # =========================================================================

    if abs(total_effect) < EPSILON:
        return pd.Series(output)


    # =========================================================================
    # %РЕЗ ИЗМЕНИЛСЯ, НО НИ ОДИН ИЗ ИЗВЕСТНЫХ ФАКТОРОВ НЕ ИЗМЕНИЛСЯ
    # =========================================================================

    if len(changed_factors) == 0:

        output["проверка"] = (
            "Изменился %рез, но изменений "
            "НИ/ПФН/НВВ/рестры/обеспеченности не найдено"
        )

        return pd.Series(output)


    # =========================================================================
    # ИЗМЕНИЛСЯ РОВНО ОДИН ФАКТОР
    # =========================================================================
    #
    # Весь эффект однозначно относится на него.
    # =========================================================================

    if len(changed_factors) == 1:

        factor = changed_factors[0]

        add_contribution(
            output=output,
            factor=factor,
            amount=total_effect,
            prev_state=prev_state,
            current_state=current_state,
        )

        validate_factor_signs(
            output,
            messages,
        )

        output["проверка"] = (
            "; ".join(messages)
            if messages
            else 0
        )

        return pd.Series(output)


    # =========================================================================
    # ИЗМЕНИЛОСЬ ДВА ИЛИ БОЛЕЕ ФАКТОРОВ
    # =========================================================================
    #
    # Используем последовательную логику переходов из исходного VBA.
    # =========================================================================

    pending_factors = []


    # Если договор ВЫШЕЛ из рестры — снимаем рестру первой.
    if (
        prev_state["REESTR"]
        and
        not current_state["REESTR"]
    ):
        pending_factors.append("REESTR")


    # Базовая последовательность обычных факторов.
    if prev_state["NI"] != current_state["NI"]:
        pending_factors.append("NI")

    if prev_state["PFN"] != current_state["PFN"]:
        pending_factors.append("PFN")

    if prev_state["NVV"] != current_state["NVV"]:
        pending_factors.append("NVV")

    if (
        normalize_scalar(prev_state["SECURITY"])
        !=
        normalize_scalar(current_state["SECURITY"])
    ):
        pending_factors.append("SECURITY")


    # Если договор ПРИШЕЛ в рестру — рестра применяется последней.
    if (
        not prev_state["REESTR"]
        and
        current_state["REESTR"]
    ):
        pending_factors.append("REESTR")


    # Начинаем со старого фактического состояния.
    state = prev_state.copy()
    state_group = prev_group
    state_rate = prev_rate

    last_factor = None


    # =========================================================================
    # ПОСЛЕДОВАТЕЛЬНО ПРИМЕНЯЕМ ФАКТОРЫ
    # =========================================================================

    while len(pending_factors) > 0:


        # Если остался один фактор — выбор очевиден.
        if len(pending_factors) == 1:

            selected_index = 0


        # Если промежуточное состояние пока находится в рестре,
        # в первую очередь пытаемся выйти из нее.
        elif state["REESTR"]:

            if "REESTR" in pending_factors:

                selected_index = (
                    pending_factors.index(
                        "REESTR"
                    )
                )

            else:

                selected_index = 0


        else:

            selected_index = (
                select_next_factor_index(
                    pending_factors=pending_factors,
                    state_group=state_group,
                    final_group=current_group,
                    state=state,
                    current_state=current_state,
                )
            )


        factor = pending_factors[
            selected_index
        ]

        last_factor = factor


        # =====================================================================
        # ПОСЛЕДНИЙ ФАКТОР
        # =====================================================================
        #
        # Он всегда доводит расчет до фактической текущей ГР и %рез.
        # Это обеспечивает равенство общей суммы факторному эффекту.
        # =====================================================================

        if len(pending_factors) == 1:

            next_group = current_group
            next_rate = current_rate


        # =====================================================================
        # ПРОМЕЖУТОЧНЫЙ ФАКТОР
        # =====================================================================

        else:

            next_group = (
                group_after_applying_factor(
                    factor=factor,
                    state=state,
                    current_state=current_state,
                )
            )


            # Если промежуточную ГР определить невозможно,
            # этот шаг получает нулевой эффект.
            if pd.isna(
                to_float(next_group)
            ):

                next_group = state_group
                next_rate = state_rate

                add_status(
                    messages,
                    (
                        "Не удалось определить промежуточную "
                        f"ГР после фактора {factor}"
                    ),
                )


            else:

                # Процент промежуточной ГР берем только из df_out.
                next_rate = (
                    get_intermediate_rate(
                        risk_group=next_group,
                        balance_type=row[
                            BALANCE_COLUMN
                        ],
                        prev_group=prev_group,
                        prev_rate=prev_rate,
                        current_group=current_group,
                        current_rate=current_rate,
                    )
                )


                # Если такая промежуточная ГР в df_out не встретилась,
                # числовой эффект этого шага остается нулевым.
                if pd.isna(next_rate):

                    add_status(
                        messages,
                        (
                            "В df_out не найден %рез "
                            f"для промежуточной ГР {next_group}"
                        ),
                    )

                    next_rate = state_rate


        # =====================================================================
        # ЭФФЕКТ ТЕКУЩЕГО ФАКТОРА
        # =====================================================================
        #
        # ВАЖНО: используется OD ПРЕДЫДУЩЕГО ПЕРИОДА.
        # =====================================================================

        contribution = (
            od
            *
            (
                state_rate
                -
                next_rate
            )
            /
            100
        )


        add_contribution(
            output=output,
            factor=factor,
            amount=contribution,
            prev_state=prev_state,
            current_state=current_state,
        )


        # Обновляем промежуточное состояние.
        state = apply_factor_to_state(
            factor=factor,
            state=state,
            current_state=current_state,
        )

        state_group = next_group
        state_rate = next_rate


        # Удаляем уже обработанный фактор.
        pending_factors.pop(
            selected_index
        )


    # =========================================================================
    # КОНТРОЛЬНЫЙ ОСТАТОК
    # =========================================================================

    factor_sum = sum(
        output[column]
        for column in FACTOR_COLUMNS
    )

    residual = (
        total_effect
        -
        factor_sum
    )


    # Остаток относим на последний фактор.
    # Это повторяет механизм контрольного выравнивания исходного алгоритма.
    if abs(residual) >= EPSILON:

        add_contribution(
            output=output,
            factor=last_factor,
            amount=residual,
            prev_state=prev_state,
            current_state=current_state,
        )

        add_status(
            messages,
            (
                f"Остаток {residual:.6f} "
                f"отнесен на последний фактор {last_factor}"
            ),
        )


    # =========================================================================
    # ФИНАЛЬНАЯ ПРОВЕРКА
    # =========================================================================

    final_factor_sum = sum(
        output[column]
        for column in FACTOR_COLUMNS
    )


    if (
        abs(
            final_factor_sum
            -
            total_effect
        )
        >=
        EPSILON
    ):

        add_status(
            messages,
            "Сумма факторов не равна ухудшению качества",
        )


    validate_factor_signs(
        output,
        messages,
    )


    output["проверка"] = (
        "; ".join(messages)
        if messages
        else 0
    )


    return pd.Series(output)



## 7. Переоценка и изменение портфеля

В исходном VBA-файле этих формул нет. Поэтому сейчас обе функции возвращают **0** для каждой строки.

Когда появятся формулы, остальной ноутбук менять не потребуется.


In [ ]:

def calculate_revaluation(
    df,
    old_period,
    new_period,
):
    """
    Переоценка.

    Пока формула не задана, возвращаем 0 для каждой строки.
    """

    return pd.Series(
        0.0,
        index=df.index,
        dtype=float,
    )


def calculate_portfolio_change(
    df,
    old_period,
    new_period,
):
    """
    Изменение портфеля.

    Пока формула не задана, возвращаем 0 для каждой строки.
    """

    return pd.Series(
        0.0,
        index=df.index,
        dtype=float,
    )



## 8. Основной расчет по всем отчетным периодам

Это и есть непосредственный **вызов расчета**. Отдельной общей функции нет.

Пары дат формируются автоматически:

`01.01 -> 01.02 -> 01.03 -> ...`

Для первой отчетной даты предыдущего периода нет, поэтому все расчетные показатели первой даты устанавливаются в **0**.


In [ ]:

# =============================================================================
# ПЕРВАЯ ОТЧЕТНАЯ ДАТА
# =============================================================================

first_period = PERIODS[0]


for factor in FACTOR_COLUMNS:

    df_out[
        make_col(
            factor,
            first_period,
        )
    ] = 0.0


df_out[
    make_col(
        "ухудшение качества",
        first_period,
    )
] = 0.0


df_out[
    make_col(
        "проверка",
        first_period,
    )
] = 0


df_out[
    make_col(
        "переоценка",
        first_period,
    )
] = 0.0


df_out[
    make_col(
        "изменение портфеля",
        first_period,
    )
] = 0.0


# =============================================================================
# ВСЕ ПОСЛЕДУЮЩИЕ ОТЧЕТНЫЕ ДАТЫ
# =============================================================================

for old_period, new_period in zip(
    PERIODS[:-1],
    PERIODS[1:],
):

    print(
        f"Расчет {old_period} -> {new_period}"
    )


    # -------------------------------------------------------------------------
    # Факторный анализ изменения качества.
    # -------------------------------------------------------------------------

    result = df_out.apply(
        calculate_quality_factors_for_row,
        axis=1,
        old_period=old_period,
        new_period=new_period,
    )


    # Записываем рассчитанные факторы и контроль.
    for column in (
        FACTOR_COLUMNS
        +
        [
            "ухудшение качества",
            "проверка",
        ]
    ):

        df_out[
            make_col(
                column,
                new_period,
            )
        ] = result[
            column
        ]


    # -------------------------------------------------------------------------
    # Переоценка.
    # Пока везде 0 до появления бизнес-формулы.
    # -------------------------------------------------------------------------

    df_out[
        make_col(
            "переоценка",
            new_period,
        )
    ] = calculate_revaluation(
        df_out,
        old_period,
        new_period,
    )


    # -------------------------------------------------------------------------
    # Изменение портфеля.
    # Пока везде 0 до появления бизнес-формулы.
    # -------------------------------------------------------------------------

    df_out[
        make_col(
            "изменение портфеля",
            new_period,
        )
    ] = calculate_portfolio_change(
        df_out,
        old_period,
        new_period,
    )


print()
print("Расчет всех периодов завершен.")



## 9. Перестановка столбцов

После расчета каждый период становится отдельным последовательным блоком:

`задолженность -> OD -> НИ -> ПФН -> НВВ -> рестра -> обеспеченность -> ГР -> %рез -> факторы -> качество -> проверка -> переоценка -> изменение портфеля`

Все постоянные клиентские поля остаются слева.


In [ ]:

PERIOD_BLOCK = [

    SOURCE_COLUMNS["debt"],

    # OD расположен сразу после задолженности.
    SOURCE_COLUMNS["od"],

    SOURCE_COLUMNS["ni"],
    SOURCE_COLUMNS["pfn"],
    SOURCE_COLUMNS["nvv"],
    SOURCE_COLUMNS["restra"],
    SOURCE_COLUMNS["security"],
    SOURCE_COLUMNS["group"],
    SOURCE_COLUMNS["rate"],

    "ушла НИ",
    "пришла НИ",

    "ушел ПФН",
    "пришел ПФН",

    "ушло НВВ",
    "пришло НВВ",

    "изменение рестры",

    "изменилась обеспеченность",

    "ухудшение качества",

    "проверка",

    "переоценка",

    "изменение портфеля",
]


# Собираем все периодические столбцы в нужном порядке.
period_columns = []


for period in PERIODS:

    for column in PERIOD_BLOCK:

        full_column = make_col(
            column,
            period,
        )

        if full_column in df_out.columns:

            period_columns.append(
                full_column
            )


# Все остальные столбцы считаем постоянными
# и оставляем в начале dataframe.
static_columns = [

    column

    for column in df_out.columns

    if column not in period_columns
]


# Физически переставляем столбцы.
df_out = df_out[
    static_columns
    +
    period_columns
].copy()



## 10. Контроль и проблемные строки

`problem_rows` содержит только те строки, где хотя бы в одном периоде в `проверка_{дата}` записано замечание.

Если ошибок нет, `problem_rows` будет пустым.


In [ ]:

problem_mask = pd.Series(
    False,
    index=df_out.index,
)


for period in PERIODS:

    check_column = make_col(
        "проверка",
        period,
    )


    # Нормальный результат проверки — числовой 0.
    # Все остальные непустые значения считаем замечанием.
    check_values = df_out[
        check_column
    ]


    period_problem_mask = ~(
        check_values.fillna(0).astype(str).str.strip().isin(
            [
                "0",
                "0.0",
                "",
            ]
        )
    )


    problem_mask |= (
        period_problem_mask
    )


problem_rows = df_out[
    problem_mask
].copy()


print("=" * 80)
print("РАСЧЕТ ЗАВЕРШЕН")
print("=" * 80)
print(f"Строк в df_out: {len(df_out):,}")
print(f"Отчетных периодов: {len(PERIODS)}")
print(f"Строк с замечаниями: {len(problem_rows):,}")
print("=" * 80)


# Показываем проблемные строки только если они есть.
if len(problem_rows) > 0:

    display(
        problem_rows
    )



## Результат

После выполнения всех ячеек:

- **`df_out`** — итоговый DataFrame со всеми расчетными столбцами в нужном порядке;
- **`problem_rows`** — строки, требующие ручной проверки;
- **`PERIODS`** — автоматически найденный список отчетных дат;
- **`rate_lookup`** — соответствия `Баланс/внебаланс + ГР -> %рез`, построенные только из самого `df_out`.

Числовые расчетные показатели, которые нельзя посчитать, остаются равными **0**. Причина при необходимости указывается в `проверка_{дата}`.
